# ThreatSense-AI — Weapon Detector Training

Trains a **YOLOv8m** weapon detector on a merged dataset of ~19k images.

**Classes:** `gun` · `knife` · `person`

---

## Before you start

1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run `scripts/zip_for_colab.py` on your local machine to create `weapon_merged.zip`
3. Upload `weapon_merged.zip` to **Google Drive → My Drive** (root)
4. Run cells top-to-bottom

The best model is saved to `My Drive/weapon_detector_best.pt` automatically.

## 1 — Check GPU

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU Info:', result.stdout.strip())

# Auto-select batch size based on available VRAM
gpu_info = result.stdout.strip()
if 'A100' in gpu_info:
    BATCH = 64
elif 'V100' in gpu_info or 'T4' in gpu_info or 'P100' in gpu_info:
    BATCH = 32
else:  # K80 or unknown
    BATCH = 16

print(f'Selected batch size: {BATCH}')

## 2 — Install dependencies

In [ ]:
!pip install ultralytics==8.4.24 -q
import ultralytics
print('ultralytics:', ultralytics.__version__)

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3 — Mount Google Drive and extract dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
ZIP_PATH     = DRIVE_ROOT / 'weapon_merged.zip'
DATASET_PATH = Path('/content/weapon_merged')

assert ZIP_PATH.exists(), (
    f'weapon_merged.zip not found at {ZIP_PATH}\n'
    'Upload it to the root of your Google Drive first.'
)

if not DATASET_PATH.exists():
    print('Extracting dataset...')
    !unzip -q "{ZIP_PATH}" -d /content/
    print('Done.')
else:
    print('Dataset already extracted.')

# Count images
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_PATH / split / 'images'
    count = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    print(f'  {split:6s}: {count:,} images')

## 4 — Write data.yaml with absolute paths

In [ ]:
import yaml

DATA_YAML = DATASET_PATH / 'data.yaml'

data_cfg = {
    'train': str(DATASET_PATH / 'train' / 'images'),
    'val':   str(DATASET_PATH / 'valid' / 'images'),
    'test':  str(DATASET_PATH / 'test'  / 'images'),
    'nc':    3,
    'names': ['gun', 'knife', 'person'],
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print('data.yaml written:')
!cat "{DATA_YAML}"

## 5 — Training configuration

Adjust these if needed. Defaults are optimised for T4 (16 GB VRAM).

In [ ]:
# ── Training config ────────────────────────────────────────────────
EPOCHS       = 150          # max epochs; early-stop at patience=20
IMGSZ        = 640          # input resolution
WORKERS      = 4            # dataloader workers
DEVICE       = 0            # GPU index (0 = first GPU)
SAVE_DIR     = '/content/runs/detect'
RUN_NAME     = 'weapon_training'

print(f'Config: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, device={DEVICE}')

## 6 — Start training

> **Expected time on T4:** ~1 min/epoch × 150 epochs ≈ **2.5 hrs** (early-stop usually cuts this to ~60–80 epochs)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data    = str(DATA_YAML),
    imgsz   = IMGSZ,
    epochs  = EPOCHS,
    batch   = BATCH,
    device  = DEVICE,
    workers = WORKERS,
    amp     = True,

    # Optimizer
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,

    # Early stopping
    patience    = 20,
    cache       = 'ram',   # RAM cache is faster than disk on Colab

    # Augmentation (real-world surveillance conditions)
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 10,
    translate    = 0.1,
    scale        = 0.5,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.1,
    auto_augment = 'randaugment',

    # Save every 10 epochs so Drive has a checkpoint if session drops
    save_period = 10,

    project  = SAVE_DIR,
    name     = RUN_NAME,
    exist_ok = True,
    verbose  = True,
)

## 7 — Validate on test set

In [ ]:
from pathlib import Path

best_pt = Path(SAVE_DIR) / RUN_NAME / 'weights' / 'best.pt'
print(f'Best weights: {best_pt}')

best_model = YOLO(str(best_pt))
metrics = best_model.val(
    data   = str(DATA_YAML),
    split  = 'test',
    device = DEVICE,
    imgsz  = IMGSZ,
)

print('\n=== Test Set Metrics ===')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50:95 : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')

## 8 — Plot training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = Path(SAVE_DIR) / RUN_NAME / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('ThreatSense-AI Training Curves', fontsize=14)

plots = [
    ('train/box_loss',  'Box Loss (train)',     axes[0, 0]),
    ('train/cls_loss',  'Class Loss (train)',   axes[0, 1]),
    ('train/dfl_loss',  'DFL Loss (train)',     axes[0, 2]),
    ('metrics/mAP50(B)',    'mAP@50',           axes[1, 0]),
    ('metrics/mAP50-95(B)', 'mAP@50:95',        axes[1, 1]),
    ('val/box_loss',    'Box Loss (val)',        axes[1, 2]),
]

for col, title, ax in plots:
    if col in df.columns:
        ax.plot(df['epoch'], df[col])
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('Plot saved to /content/training_curves.png')

## 9 — Save model to Google Drive

In [ ]:
import shutil
from pathlib import Path

best_pt   = Path(SAVE_DIR) / RUN_NAME / 'weights' / 'best.pt'
last_pt   = Path(SAVE_DIR) / RUN_NAME / 'weights' / 'last.pt'
drive_dst = Path('/content/drive/MyDrive')

shutil.copy2(best_pt, drive_dst / 'weapon_detector_best.pt')
shutil.copy2(last_pt, drive_dst / 'weapon_detector_last.pt')
shutil.copy2('/content/training_curves.png', drive_dst / 'training_curves.png')

# Also copy the full results CSV
results_csv = Path(SAVE_DIR) / RUN_NAME / 'results.csv'
shutil.copy2(results_csv, drive_dst / 'weapon_training_results.csv')

print('Saved to Google Drive:')
print(f'  weapon_detector_best.pt  ({best_pt.stat().st_size / 1e6:.1f} MB)')
print(f'  weapon_detector_last.pt')
print(f'  training_curves.png')
print(f'  weapon_training_results.csv')

## 10 — (Optional) Resume if session disconnected

If your Colab session expired mid-training, run this cell instead of cell 6.
It will continue from the last saved checkpoint.

In [ ]:
# ── RESUME CELL — only run this if the session disconnected ──────
from ultralytics import YOLO
from pathlib import Path

last_pt = Path(SAVE_DIR) / RUN_NAME / 'weights' / 'last.pt'

if not last_pt.exists():
    print('No checkpoint found. Run cell 6 to start training from scratch.')
else:
    print(f'Resuming from {last_pt}')
    model = YOLO(str(last_pt))
    results = model.train(resume=True)

## 11 — (Optional) Quick inference test

Test the trained model on a sample image to visually verify detection quality.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import glob, random
from IPython.display import Image, display

best_pt = Path(SAVE_DIR) / RUN_NAME / 'weights' / 'best.pt'
model   = YOLO(str(best_pt))

# Pick 3 random test images
test_images = glob.glob(str(DATASET_PATH / 'test' / 'images' / '*.jpg'))
samples = random.sample(test_images, min(3, len(test_images)))

results = model.predict(
    source=samples,
    conf=0.40,
    save=True,
    project='/content/inference',
    name='test_samples',
    exist_ok=True,
)

# Display results inline
out_imgs = sorted(glob.glob('/content/inference/test_samples/*.jpg'))
for img_path in out_imgs[:3]:
    display(Image(filename=img_path, width=640))